# Stage 10 (repurposed) — General F&O trend-following 1D status (live refresh, full universe)

**REPURPOSED** (see `trend_universe.py` and README Stage 10 for the
history): the universe is now the full F&O stock+index universe, not the
154-symbol pledgeable-securities list. The strategy is unchanged in spirit —
buy on a fresh 1D Supertrend UP flip, hold, exit on the opposite DOWN flip —
but it's now literal cash-equity ownership for any of these names, not
specifically about monitoring pledged collateral. **No stop-loss**:
`src/stop_loss_test.py` backtested fixed-%, ATR-multiple, and worst-day-based
candidate stop rules against real 1D flip history and found none beat holding
to the natural DOWN-flip exit — every rule's stopped-out flips would have
averaged a POSITIVE outcome if held (consistent with `reversion_test.py`'s
momentum/continuation finding). Exit is the Supertrend DOWN flip only, by
design, not by omission — `buffer_to_flip_pct`/`moved_since_entry_pct` below
remain the informational "how close/how underwater" numbers.

**Corrected design (kept from the original build):** a purely read-only
version can go stale the same way the credit-spread screener's already-
documented gap works - if a symbol flipped after the last fetch, the saved
report keeps showing the old direction/age. This version genuinely refreshes
every symbol before reporting status.

**Second correction - a real regression found while building the Stage 11
pledged-trend-following ranking:** the original version of this cell called
`run_all_instruments(..., force_refresh=True)` with a short 390-day window,
which does a full REPLACE of each symbol's saved raw data, not an update.
Every routine live-screener run was silently throwing away notebook 10's deep
multi-year history and replacing it with just that short window - after
repeated runs, every symbol's saved history had collapsed to ~1 year, and
UP-closed-flip counts per symbol fell to 1-8, nowhere near usable for a
ranking. Fixed by switching to `incremental=True`
(`process_instrument_incremental()` in `trend_1d_pipeline.py`): it EXTENDS
existing saved history with just the recent catch-up window instead of
replacing it outright, so routine checks no longer erode the deep history
notebook 10 built. `from_date` is no longer meaningful in this mode (each
symbol's own last-saved date drives its fetch window) - only `to_date` and
`delay_seconds` still apply.

**Rate limit hit three times in a row** even after the chunk-pacing fix in
`fetch_data.py` and a 1-second between-symbol delay - close-to-the-limit
pacing (~0.35-0.4s spacing, right at the documented 3 req/sec ceiling) isn't
leaving enough real-world margin for network jitter, and repeated failures in
the same session may also mean Kite briefly throttles further after tripping
the limit once. `delay_seconds=1.0` between symbols - real headroom, not the
minimum. If this still fails, it's worth waiting a minute or two before
retrying, in case of a lingering cooldown from the earlier attempts.

## Column reference

**Is the trend intact?**

| Column | Meaning |
|---|---|
| `direction` | The currently OPEN 1D Supertrend flip |
| `trend_age_days` | Calendar days since that flip |
| `spot` · `flip_level` | Latest close, and the Supertrend level that would reverse the trend |
| `buffer_to_flip_pct` | Distance between them. **Small = about to invalidate.** With no stop-loss (see below), this is the single most actionable "how close to the exit" number here |
| `moved_since_entry_pct` | Signed *for* the trend: positive = working, negative = underwater |

**Does the signal actually work on this instrument?**

| Column | Meaning |
|---|---|
| `dir_flips` | Closed flips on the currently open side. Under 10, the edge columns are blank |
| `dir_win_rate` | % of those that moved the way the signal called |
| `dir_median_move` | Median favourable move. Negative = the typical flip goes the *wrong* way |
| `signal_works_here` | `dir_win_rate >= 50` **and** `dir_median_move > 0` |
| `opp_win_rate` · `opp_median_move` | The other direction, for contrast only |

> **Why this matters on this universe.** Measured on the original 154-symbol
> pledged universe (100 instruments with enough history): UP flips won 54.1%
> (+0.9% mean median move, 68% show an edge). DOWN flips won **42.4%** with a
> **−1.3%** mean median move and only **14%** show an edge — the median DOWN
> flip ends with price *higher* than where it flipped. Worth re-checking this
> split on the full F&O universe once enough calendar time has passed, but
> UP-only scoring is the right default in the meantime.

**No stop-loss.** `src/stop_loss_test.py` tested fixed-% (5/8/10/12%),
ATR-multiple (1.5×/2×/2.5×), and worst-day-based candidate stop rules against
real closed UP flips — every single one reduced mean/median return and win
rate versus simply holding to the natural DOWN-flip exit, because stopped-out
flips consistently would have recovered to a profit if held (consistent with
`reversion_test.py`'s momentum/continuation finding, not reversion). Exit is
the Supertrend DOWN flip only.

**Risk / event context**

| Column | Meaning |
|---|---|
| `worst_day` | Largest single close-to-close day against the position, 4yr |
| `entry_day_move` | Move on the day this flip was born |
| `born_on_outlier_day` | `abs(entry_day_move) > 4%` — a flip created by a one-day collapse is a different event, and the duration stats come from ordinary flips |

**Duration context** — descriptive only

| Column | Meaning |
|---|---|
| `median_duration` | This instrument's typical 1D flip length |
| `past_median` · `past_75th_pct` · `past_90th_pct` | Has the open trend outlived that checkpoint? |
| `runway_left_vs_median` · `_p75` · `_p90` | `checkpoint − age` in signed days; negative = past it |

> `reversion_test.py` and `exhaustion_test.py` both found that crossing a duration
> checkpoint does **not** predict a reversal. Not a warning, not an exit signal.

**Universe & score** *(from `trend_universe.py` and notebook 12)*

| Column | Meaning |
|---|---|
| `security_type` | `Stock` or `Index` — display/filter only, not a separate scoring scale |
| `trend_score` | UP-trend-following rank from notebook 12. Blank = thin sample, not a low score |
| `thin_sample` | Fewer than 15 UP-closed flips, so unscored |
| `outlier_reliance_flag` · `gap_risk_flag` | Score depends on a few outlier wins / high gap history |

*All statistics are in-sample and descriptive — what the signal HAS done, not what it will do.*

In [1]:
import sys
sys.path.append("..")

from datetime import datetime, timedelta
from src.auth import get_kite
from src.instruments import build_nse_token_lookup, get_index_token_lookup
from src.trend_universe import load_trend_universe, save_trend_universe
from src.trend_1d_pipeline import run_all_instruments

kite = get_kite()
universe = load_trend_universe(kite=kite)
save_trend_universe(fno_symbol_list=universe[universe.security_type == "Stock"]["symbol"].tolist())
symbol_list = universe["symbol"].tolist()

token_lookup = build_nse_token_lookup(kite)
token_lookup.update(get_index_token_lookup(kite))

# incremental=True - EXTENDS each symbol's existing saved history with just
# the recent catch-up window (per-symbol, derived from its own last-saved
# date) instead of replacing it outright. from_date is unused in this mode,
# kept only because run_all_instruments still accepts it positionally.
#
# delay_seconds bumped well past the 3 req/sec limit's minimum spacing
# (~0.33s) - three failed attempts in a row means it's worth real headroom
# over cutting it close, even though it makes the full run slower.
results = run_all_instruments(
    kite, token_lookup, symbol_list,
    from_date=None,
    to_date=datetime.now(),
    incremental=True,
    delay_seconds=1.0
)
results["status"].value_counts()

Saved 217 symbols (211 stocks, 6 indices) to /Users/hgmac/Documents/3_trading/Quant/config/trend_universe.csv
[1/217] 360ONE — processing...
Fetching 2025-07-28 to 2026-08-27...
Fetching 2026-08-28 to 2026-09-01...
Saved 1981 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/360ONE_day.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/trend_1d/360ONE_full_report.xlsx
[1/217] 360ONE — done
[2/217] ABB — processing...
Fetching 2025-07-28 to 2026-08-27...
Fetching 2026-08-28 to 2026-09-01...
Saved 2887 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/ABB_day.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/trend_1d/ABB_full_report.xlsx
[2/217] ABB — done
[3/217] ABCAPITAL — processing...
Fetching 2025-07-28 to 2026-08-27...
Fetching 2026-08-28 to 2026-09-01...
Saved 2229 candles to /Users/hgmac/Documents/3_trading/Quant/data/raw/ABCAPITAL_day.csv
Saved full report to /Users/hgmac/Documents/3_trading/Quant/exports/trend_1d/ABC

status
SUCCESS    216
FAILED       1
Name: count, dtype: int64

In [2]:
import pandas as pd

# Every table below is wide (30 columns after the trend_score merge)
# and the Shares table alone is 100+ rows — comfortably past pandas' defaults
# (60 rows, and columns wrap rather than showing all 30 side by side), so both
# axes were truncating with "..." in the middle. Nothing was ever missing from
# the underlying data, just not printed. max_columns=None shows every column;
# width=250 stops them wrapping; max_rows=250 covers the largest table here.
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 250)

In [3]:
from src.trend_1d_screener import check_trend_status
import pandas as pd

results = [check_trend_status(s, universe_df=universe) for s in symbol_list]
results = [r for r in results if r is not None]
status = pd.DataFrame(results).sort_values("trend_age_days")
print("SORTED BY: trend_age_days ascending — sets the order the status_up/top25 tables inherit")
print(f"{len(status)} of {len(symbol_list)} instruments have a current status")

No report found for NIFTYFPI - refresh it first
SORTED BY: trend_age_days ascending — sets the order the status_up/top25 tables inherit
216 of 217 instruments have a current status


In [4]:
# Merge in score_for_trend_following()'s ranking (Stage 10/11) -
# answers "which of several same-day UP flips should I prioritize" instead
# of leaving it a guess. Run notebook 12 first if these files don't exist
# yet or look stale.
ranking = pd.read_excel("../exports/RANKING_trend_following.xlsx")
watchlist = pd.read_excel("../exports/WATCHLIST_trend_following.xlsx")

score_cols = ["symbol", "trend_score", "outlier_reliance_flag", "gap_risk_flag"]
ranking_slim = ranking[score_cols]

# Watchlist symbols have no score yet (below the 15 UP-closed-flip floor) -
# kept as NaN/blank rather than 0, so "no score" is never confused with
# "scored low". thin_sample flags which case applies.
watchlist_slim = watchlist[["symbol"]].copy()
for col in ["trend_score", "outlier_reliance_flag", "gap_risk_flag"]:
    watchlist_slim[col] = None
watchlist_slim["thin_sample"] = True

score_lookup = pd.concat([
    ranking_slim.assign(thin_sample=False),
    watchlist_slim
], ignore_index=True)

status = status.merge(score_lookup, on="symbol", how="left")
print(f"Scored: {status['trend_score'].notna().sum()} of {len(status)} — rest are thin-sample (< 15 UP-closed flips)")

KeyError: "['trend_score'] not in index"

## All instruments — current status

In [ ]:
print("SORTED BY: trend_age_days ascending — most recently flipped first")
status

SORTED BY: trend_age_days ascending — most recently flipped first


,symbol,direction,trend_age_days,median_duration,past_median,past_75th_pct,past_90th_pct,runway_left_vs_median,runway_left_vs_p75,runway_left_vs_p90,dir_flips,dir_win_rate,dir_median_move,worst_day,opp_win_rate,opp_median_move,signal_works_here,spot,flip_level,buffer_to_flip_pct,moved_since_entry_pct,entry_day_move,born_on_outlier_day,security_type,trend_score,outlier_reliance_flag,gap_risk_flag,thin_sample
0,KAYNES,DOWN,0,53.0,False,False,False,53.0,78.0,113.0,10,50.0,-2.42,13.76,81.8,10.86,False,3635.20,4093.08,12.60,-0.00,-1.35,False,Stock,None,None,None,True
1,MARUTI,DOWN,0,39.0,False,False,False,39.0,66.5,115.4,39,43.6,-2.19,13.51,65.0,2.32,False,12926.00,13987.91,8.22,-0.00,-4.58,True,Stock,78.5,⚠ RELIES ON OUTLIER WINS (avg-median gap 3.22pp),NaN,False
2,M&M,DOWN,0,41.0,False,False,False,41.0,61.0,98.8,39,43.6,-1.45,14.36,52.5,0.28,False,3264.20,3516.32,7.72,-0.00,-0.54,False,Stock,47.3,⚠ RELIES ON OUTLIER WINS (avg-median gap 5.73pp),NaN,False
3,MUTHOOTFIN,DOWN,0,49.0,False,False,False,49.0,79.5,125.2,33,45.5,-0.37,16.73,55.9,3.16,False,2930.50,3241.34,10.61,-0.00,-1.60,False,Stock,60.8,⚠ RELIES ON OUTLIER WINS (avg-median gap 6.33pp),NaN,False
4,NIFTYNXT50,DOWN,0,32.0,False,False,False,32.0,51.0,74.8,51,37.3,-1.59,6.39,61.5,0.92,False,73055.90,75065.96,2.75,-0.00,-0.95,False,Index,73.2,⚠ RELIES ON OUTLIER WINS (avg-median gap 3.16pp),NaN,False
5,ADANIENT,DOWN,1,46.0,False,False,False,45.0,63.5,107.6,37,48.6,-0.60,20.04,50.0,0.60,False,2864.10,3160.04,10.33,-0.17,-9.76,True,Stock,26.7,⚠ RELIES ON OUTLIER WINS (avg-median gap 33.74pp),⚠ HIGH GAP RISK (>10.0%),False
6,BAJFINANCE,DOWN,1,42.0,False,False,False,41.0,70.0,100.0,38,36.8,-2.49,11.06,69.2,7.79,False,1057.30,1122.95,6.21,-0.03,-2.12,False,Stock,91.7,⚠ RELIES ON OUTLIER WINS (avg-median gap 4.33pp),NaN,False
7,OFSS,UP,1,46.0,False,False,False,45.0,78.2,126.9,34,41.2,-2.24,8.31,41.2,-1.16,False,12220.00,11257.20,7.88,-2.55,2.87,False,Stock,14.2,⚠ RELIES ON OUTLIER WINS (avg-median gap 9.77pp),NaN,False
8,BRITANNIA,DOWN,1,50.0,False,False,False,49.0,86.0,107.2,34,41.2,-1.08,10.61,57.1,2.41,False,5169.50,5511.20,6.61,1.53,-1.10,False,Stock,73.3,⚠ RELIES ON OUTLIER WINS (avg-median gap 4.05pp),NaN,False
9,AXISBANK,UP,1,54.0,False,False,False,53.0,87.0,112.7,34,67.6,3.66,7.57,38.2,-1.36,True,1266.30,1209.49,4.49,-2.59,2.77,False,Stock,87.8,⚠ RELIES ON OUTLIER WINS (avg-median gap 3.40pp),NaN,False


In [ ]:
status_up = status[status["direction"] == "UP"].sort_values("trend_age_days", ascending=True).head(150).reset_index(drop=True)

print("SORTED BY: trend_age_days ascending — most recently flipped UP first")
status_up

SORTED BY: trend_age_days ascending — most recently flipped UP first


,symbol,direction,trend_age_days,median_duration,past_median,past_75th_pct,past_90th_pct,runway_left_vs_median,runway_left_vs_p75,runway_left_vs_p90,dir_flips,dir_win_rate,dir_median_move,worst_day,opp_win_rate,opp_median_move,signal_works_here,spot,flip_level,buffer_to_flip_pct,moved_since_entry_pct,entry_day_move,born_on_outlier_day,security_type,trend_score,outlier_reliance_flag,gap_risk_flag,thin_sample
0,OFSS,UP,1,46.0,False,False,False,45.0,78.2,126.9,34,41.2,-2.24,8.31,41.2,-1.16,False,12220.00,11257.20,7.88,-2.55,2.87,False,Stock,14.2,⚠ RELIES ON OUTLIER WINS (avg-median gap 9.77pp),NaN,False
1,AXISBANK,UP,1,54.0,False,False,False,53.0,87.0,112.7,34,67.6,3.66,7.57,38.2,-1.36,True,1266.30,1209.49,4.49,-2.59,2.77,False,Stock,87.8,⚠ RELIES ON OUTLIER WINS (avg-median gap 3.40pp),NaN,False
2,SUNPHARMA,UP,1,45.5,False,False,False,44.5,70.8,122.2,37,51.4,0.27,6.39,48.6,-0.18,True,1931.30,1839.93,4.73,-2.70,3.37,False,Stock,47.7,⚠ RELIES ON OUTLIER WINS (avg-median gap 3.78pp),NaN,False
3,CGPOWER,UP,1,45.5,False,False,False,44.5,67.2,97.7,40,57.5,2.99,8.63,40.0,-2.53,True,885.95,827.70,6.57,-3.28,3.04,False,Stock,54.7,⚠ RELIES ON OUTLIER WINS (avg-median gap 14.14pp),NaN,False
4,AMBER,UP,5,50.0,False,False,False,45.0,75.8,96.8,24,45.8,-1.18,10.28,50.0,-0.26,False,7582.50,7160.28,5.57,-1.54,3.15,False,Stock,15.5,⚠ RELIES ON OUTLIER WINS (avg-median gap 15.92pp),⚠ HIGH GAP RISK (>10.0%),False
5,CDSL,UP,6,36.5,False,False,False,30.5,71.2,128.5,28,39.3,-3.49,9.99,46.4,-0.28,False,1386.90,1329.38,4.15,-3.28,2.81,False,Stock,5.5,⚠ RELIES ON OUTLIER WINS (avg-median gap 15.71pp),NaN,False
6,MAHABANK,UP,6,41.0,False,False,False,35.0,58.2,103.5,38,47.4,-1.69,8.97,42.1,-2.36,False,82.85,78.51,5.24,-3.21,4.77,True,Stock,19.1,⚠ RELIES ON OUTLIER WINS (avg-median gap 6.12pp),NaN,False
7,LICHSGFIN,UP,6,38.0,False,False,False,32.0,65.5,91.0,41,43.9,-0.29,9.49,48.8,-0.42,False,534.40,500.41,6.36,1.91,3.64,False,Stock,33.6,⚠ RELIES ON OUTLIER WINS (avg-median gap 3.40pp),NaN,False
8,NMDC,UP,6,43.0,False,False,False,37.0,72.2,113.8,36,66.7,3.70,10.08,38.9,-4.27,True,85.96,81.41,5.30,-2.55,2.64,False,Stock,71.4,⚠ RELIES ON OUTLIER WINS (avg-median gap 8.03pp),NaN,False
9,HDFCAMC,UP,7,33.5,False,False,False,26.5,56.8,102.3,29,48.3,-0.25,6.33,48.3,-0.84,False,2551.30,2517.52,1.32,-5.44,3.06,False,Stock,33.7,⚠ RELIES ON OUTLIER WINS (avg-median gap 7.67pp),NaN,False


## Top 25 UP-trending, ranked by trend_score (best first)

Sorted by score, not age — a fresh flip isn't automatically the best pick, and this is the view meant to answer that question directly when several instruments flip UP the same day. `trend_age_days` is still shown for context. Thin-sample names (score is blank) are included but sort to the bottom, not silently dropped.

## Good buy candidates today — the actual daily shortlist

Requires `signal_works_here == True` explicitly — UP has to have
HISTORICALLY WORKED on that specific instrument, not merely be the direction
currently showing. That is the one column from this session's work that
separates a real signal from noise (see the column reference at the top:
DOWN flips work on only 14% of this universe, UP on 68% — but even within
UP, `signal_works_here` still filters out the instruments where it does not
hold). Trend age is NOT part of the filter — a trend that flipped three
weeks ago and is still intact is just as buyable as one from yesterday
(`reversion_test.py`/`exhaustion_test.py` found age doesn't predict anything).

`thin_sample` names (fewer than 15 UP-closed flips → no `trend_score`)
are excluded here rather than sorted to the bottom — with no score, there's no
ranking basis, and this is meant to be a clean go-to list, not a place to
present unranked guesses next to ranked ones.

In [ ]:
good_buy_cols = ["symbol", "security_type", "trend_age_days", "trend_score", "dir_win_rate",
                  "dir_median_move", "buffer_to_flip_pct", "moved_since_entry_pct",
                  "worst_day", "outlier_reliance_flag", "gap_risk_flag"]

good_buy_candidates = status[
    (status["direction"] == "UP") &
    (status["signal_works_here"] == True) &
    (status["thin_sample"] == False)
].sort_values("trend_score", ascending=False)[good_buy_cols].reset_index(drop=True)

print(f"{len(good_buy_candidates)} instrument(s) — UP, signal historically works here, ranked")
print("SORTED BY: trend_score descending")
good_buy_candidates

86 instrument(s) — UP, signal historically works here, ranked
SORTED BY: trend_score descending


,symbol,security_type,trend_age_days,trend_score,dir_win_rate,dir_median_move,buffer_to_flip_pct,moved_since_entry_pct,worst_day,outlier_reliance_flag,gap_risk_flag
0,AXISBANK,Stock,1,87.8,67.6,3.66,4.49,-2.59,7.57,⚠ RELIES ON OUTLIER WINS (avg-median gap 3.40pp),NaN
1,BAJAJHLDNG,Stock,146,87.1,70.0,4.10,4.95,14.05,7.45,⚠ RELIES ON OUTLIER WINS (avg-median gap 3.85pp),NaN
2,MIDCPNIFTY,Index,75,84.7,65.9,2.11,1.47,1.87,4.91,NaN,NaN
3,LT,Stock,29,83.8,65.0,2.70,1.90,-0.19,5.91,NaN,NaN
4,TITAN,Stock,77,83.4,62.9,4.65,2.38,16.38,6.73,⚠ RELIES ON OUTLIER WINS (avg-median gap 4.74pp),NaN
5,FINNIFTY,Index,29,83.0,63.0,2.36,0.26,-3.53,3.37,NaN,NaN
6,BOSCHLTD,Stock,152,81.6,64.5,3.40,4.41,50.02,5.89,NaN,NaN
7,TATASTEEL,Stock,27,79.6,65.8,3.76,2.59,-4.12,8.57,⚠ RELIES ON OUTLIER WINS (avg-median gap 5.92pp),NaN
8,HINDALCO,Stock,28,79.5,64.9,4.37,2.32,0.13,10.98,⚠ RELIES ON OUTLIER WINS (avg-median gap 5.15pp),NaN
9,NESTLEIND,Stock,57,79.4,57.6,2.99,1.20,-1.97,5.44,NaN,NaN


### Top 25 UP-trending, ranked by score

In [ ]:
top25_up = status_up[status_up["direction"] == "UP"].sort_values(
    "trend_score", ascending=False, na_position="last"
).head(25)
print("SORTED BY: trend_score descending, thin-sample (blank score) last")
top25_up

SORTED BY: trend_score descending, thin-sample (blank score) last


,symbol,direction,trend_age_days,median_duration,past_median,past_75th_pct,past_90th_pct,runway_left_vs_median,runway_left_vs_p75,runway_left_vs_p90,dir_flips,dir_win_rate,dir_median_move,worst_day,opp_win_rate,opp_median_move,signal_works_here,spot,flip_level,buffer_to_flip_pct,moved_since_entry_pct,entry_day_move,born_on_outlier_day,security_type,trend_score,outlier_reliance_flag,gap_risk_flag,thin_sample
1,AXISBANK,UP,1,54.0,False,False,False,53.0,87.0,112.7,34,67.6,3.66,7.57,38.2,-1.36,True,1266.30,1209.49,4.49,-2.59,2.77,False,Stock,87.8,⚠ RELIES ON OUTLIER WINS (avg-median gap 3.40pp),NaN,False
116,BAJAJHLDNG,UP,146,59.5,True,True,True,-86.5,-55.0,-10.7,30,70.0,4.10,7.45,46.7,-0.92,True,11371.00,10807.72,4.95,14.05,7.94,True,Stock,87.1,⚠ RELIES ON OUTLIER WINS (avg-median gap 3.85pp),NaN,False
75,MIDCPNIFTY,UP,75,15.0,True,True,True,-60.0,-48.0,-26.0,91,65.9,2.11,4.91,42.9,-0.45,True,14868.50,14650.35,1.47,1.87,0.14,False,Index,84.7,NaN,NaN,False
34,LT,UP,29,36.0,False,False,False,7.0,38.8,78.0,40,65.0,2.70,5.91,52.5,0.04,True,4017.50,3940.99,1.90,-0.19,2.19,False,Stock,83.8,NaN,NaN,False
81,TITAN,UP,77,41.0,True,False,False,-36.0,0.8,37.4,35,62.9,4.65,6.73,37.1,-2.58,True,5048.50,4928.12,2.38,16.38,1.27,False,Stock,83.4,⚠ RELIES ON OUTLIER WINS (avg-median gap 4.74pp),NaN,False
37,FINNIFTY,UP,29,33.5,False,False,False,4.5,30.2,73.3,46,63.0,2.36,3.37,43.5,-1.33,True,26030.90,25963.27,0.26,-3.53,1.47,False,Index,83.0,NaN,NaN,False
120,BOSCHLTD,UP,152,45.5,True,True,True,-106.5,-65.0,-20.6,31,64.5,3.40,5.89,64.5,1.91,True,48210.00,46082.99,4.41,50.02,4.90,True,Stock,81.6,NaN,NaN,False
28,TATASTEEL,UP,27,43.5,False,False,False,16.5,44.2,80.0,38,65.8,3.76,8.57,42.1,-1.62,True,185.04,180.25,2.59,-4.12,1.07,False,Stock,79.6,⚠ RELIES ON OUTLIER WINS (avg-median gap 5.92pp),NaN,False
32,HINDALCO,UP,28,43.0,False,False,False,15.0,46.2,87.9,37,64.9,4.37,10.98,35.1,-2.35,True,1021.30,997.57,2.32,0.13,2.52,False,Stock,79.5,⚠ RELIES ON OUTLIER WINS (avg-median gap 5.15pp),NaN,False
68,NESTLEIND,UP,57,47.5,True,False,False,-9.5,33.8,70.5,33,57.6,2.99,5.44,39.4,-1.62,True,1444.60,1427.21,1.20,-1.97,0.95,False,Stock,79.4,NaN,NaN,False


In [12]:
top25_up_by_age = top25_up.sort_values("trend_age_days", ascending=True)
print("SORTED BY: trend_age_days ascending — same 25 shares as top25_up, just re-ordered by age")
top25_up_by_age

SORTED BY: trend_age_days ascending — same 25 shares as top25_up, just re-ordered by age


,symbol,direction,trend_age_days,median_duration,past_median,past_75th_pct,past_90th_pct,runway_left_vs_median,runway_left_vs_p75,runway_left_vs_p90,dir_flips,dir_win_rate,dir_median_move,worst_day,opp_win_rate,opp_median_move,signal_works_here,spot,flip_level,buffer_to_flip_pct,moved_since_entry_pct,entry_day_move,born_on_outlier_day,security_type,trend_score,outlier_reliance_flag,gap_risk_flag,thin_sample
2,AXISBANK,UP,1,54.0,False,False,False,53.0,87.0,112.7,34,67.6,3.66,7.57,38.2,-1.36,True,1257.50,1209.49,3.82,-3.27,2.77,False,Stock,87.8,⚠ RELIES ON OUTLIER WINS (avg-median gap 3.40pp),NaN,False
8,NMDC,UP,6,43.0,False,False,False,37.0,72.2,113.8,36,66.7,3.70,10.08,38.9,-4.27,True,85.45,81.41,4.73,-3.13,2.64,False,Stock,71.4,⚠ RELIES ON OUTLIER WINS (avg-median gap 8.03pp),NaN,False
17,BEL,UP,19,38.0,False,False,False,19.0,43.0,83.0,41,53.7,3.72,7.67,34.1,-4.01,True,410.50,394.10,4.00,0.00,1.18,False,Stock,69.8,⚠ RELIES ON OUTLIER WINS (avg-median gap 3.72pp),NaN,False
27,TATASTEEL,UP,27,43.5,False,False,False,16.5,44.2,80.0,38,65.8,3.76,8.57,42.1,-1.62,True,183.16,180.25,1.59,-5.09,1.07,False,Stock,79.6,⚠ RELIES ON OUTLIER WINS (avg-median gap 5.92pp),NaN,False
30,HINDALCO,UP,28,43.0,False,False,False,15.0,46.2,87.9,37,64.9,4.37,10.98,35.1,-2.35,True,1014.40,997.57,1.66,-0.55,2.52,False,Stock,79.5,⚠ RELIES ON OUTLIER WINS (avg-median gap 5.15pp),NaN,False
29,JSWSTEEL,UP,28,43.5,False,False,False,15.5,46.2,79.3,39,61.5,3.30,8.49,25.6,-4.36,True,1314.00,1265.27,3.71,1.08,0.60,False,Stock,75.8,NaN,NaN,False
32,LT,UP,29,36.0,False,False,False,7.0,38.8,78.0,40,65.0,2.70,5.91,52.5,0.04,True,3999.00,3940.99,1.45,-0.65,2.19,False,Stock,83.4,NaN,NaN,False
35,FINNIFTY,UP,29,33.5,False,False,False,4.5,30.2,73.3,46,63.0,2.36,3.37,43.5,-1.33,True,26003.90,25963.27,0.16,-3.63,1.47,False,Index,82.8,NaN,NaN,False
36,AUBANK,UP,29,55.0,False,False,False,26.0,58.5,87.9,24,70.8,4.54,6.31,58.3,1.06,True,1080.00,1053.42,2.46,1.89,1.21,False,Stock,74.1,⚠ RELIES ON OUTLIER WINS (avg-median gap 7.94pp),NaN,False
41,JINDALSTEL,UP,32,42.0,False,False,False,10.0,45.0,69.5,38,71.1,6.52,10.29,36.8,-2.78,True,1148.00,1106.25,3.64,4.14,0.80,False,Stock,78.6,⚠ RELIES ON OUTLIER WINS (avg-median gap 6.48pp),NaN,False
